# OpenCode Free-Model Proxy — Bun Testbed (Colab)

Runs our Bun/TypeScript stack (`fingerprint.ts`, `hit-model.ts`, `proxy.ts`, `minimize-system.ts`) against OpenCode's free models from a **fresh Colab egress IP** — the anonymous free quota is per-IP, so this is also the recovery path when the home IP is burned.

**Layout — run top to bottom:**
1. **Setup** — install Bun, download the project files from GitHub
2. **Tests** — one cell: model list → two model hits → proxy smoke test
3. **Option A: Cloudflare tunnel** — public URL, random every start
4. **Option B: zrok tunnel** — fixed URL from the `ZROK_NAME` secret, default `oc-llm-proxy.shares.zrok.io` (recommended)
5. **Ops** — quota/new-session recovery + stop everything

⚠️ Each inference test spends a little of this IP's free quota. Don't spam.

## 1 · Setup — Bun + project files

Install Bun, then run the download cell — it pulls the project files (`src/*.ts`, `data/opencode-system.txt`, `package.json`) from the public GitHub repo, so this notebook always runs the latest committed code.

In [ ]:
import os, subprocess
# Install Bun (Colab has no Bun by default)
if not os.path.exists(os.path.expanduser('~/.bun/bin/bun')):
    subprocess.run('curl -fsSL https://bun.sh/install | bash', shell=True, check=True)
os.environ['PATH'] += os.pathsep + os.path.expanduser('~/.bun/bin')
print(subprocess.run(['bun', '--version'], capture_output=True, text=True).stdout)

In [ ]:
# Download the project files from GitHub (public repo - always the latest committed code)
import os, urllib.request
BASE = 'https://raw.githubusercontent.com/recursive-factory/opencode-proxy/main/'
FILES = ['src/fingerprint.ts', 'src/hit-model.ts', 'src/proxy.ts', 'src/minimize-system.ts',
         'data/opencode-system.txt', 'package.json']
os.makedirs('/content/opencode-proxy', exist_ok=True)
for f in FILES:
    dst = '/content/opencode-proxy/' + os.path.basename(f)
    urllib.request.urlretrieve(BASE + f, dst)
    print(os.path.basename(f), os.path.getsize(dst), 'bytes')

## 2 · Tests

One cell runs the whole suite: model list (free, no quota) → one hit on two different models → proxy smoke test (`/health`, `/v1/models`, one chat). Each hit spends a little of this Colab IP's free quota — that's the point of the test.

*(Optional, ~11 calls: `bun run minimize-system.ts` finds the smallest system prompt that still passes. Only run it when you mean it.)*

In [ ]:
# Tests - one cell runs the whole suite. Each hit spends a little of this IP's free quota.
import json, os, subprocess, time, urllib.request

env = dict(os.environ)
env['PATH'] += os.pathsep + os.path.expanduser('~/.bun/bin')

def bun(args):
    p = subprocess.run(args, cwd='/content/opencode-proxy', env=env, capture_output=True, text=True, timeout=600)
    print(p.stdout[-3000:])
    if p.returncode != 0:
        print('STDERR:', p.stderr[-1000:], '\nexit code:', p.returncode)
    return p

print('== 1/4  Free models (no quota spent) ==')
bun(['bun', 'run', 'hit-model.ts', '--list'])

print('== 2/4  Hit big-pickle ==')
bun(['bun', 'run', 'hit-model.ts', 'big-pickle', 'Say hi in exactly five words.'])

print('== 3/4  Hit deepseek (different model = different identity block) ==')
bun(['bun', 'run', 'hit-model.ts', 'deepseek-v4.1-flash-free', 'Say hi in exactly five words.'])

print('== 4/4  Proxy smoke test ==')
base = 'http://127.0.0.1:8787'
proc = None
try:
    print('health:', urllib.request.urlopen(base + '/health', timeout=2).read().decode())
    print('(proxy already running - smoke-testing the live one)')
except Exception:
    proc = subprocess.Popen(['bun', 'run', 'proxy.ts'], cwd='/content/opencode-proxy', env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for _ in range(60):
        try:
            print('health:', urllib.request.urlopen(base + '/health', timeout=2).read().decode())
            break
        except Exception:
            time.sleep(1)
print('models:', urllib.request.urlopen(base + '/v1/models', timeout=10).read().decode()[:400])
req = urllib.request.Request(base + '/v1/chat/completions',
    data=json.dumps({'model': 'big-pickle',
                     'messages': [{'role': 'user', 'content': 'Say hi in exactly five words.'}]}).encode(),
    headers={'Content-Type': 'application/json'})
print('chat:', urllib.request.urlopen(req, timeout=180).read().decode()[:800])
if proc:
    proc.terminate()
print('== tests done ==')

## 3 · Option A — Cloudflare quick tunnel (random URL)

Exposes the Colab proxy at a random `*.trycloudflare.com` URL. **No API key** — the unguessable URL is the only protection; don't share it (anyone with it can spend this IP's free quota).

**Client config:** base URL = `PUBLIC_URL + /v1`, API key = anything (e.g. `not-needed`). For Cursor/VS Code the model URL is the full `PUBLIC_URL/v1/chat/completions`.

⚠️ New random URL every start, and it dies with the runtime. Prefer Option B (zrok) for a stable address.

In [ ]:
# Cloudflare · expose the proxy through a quick tunnel (no API key - URL secrecy only)
import os, re, subprocess, time, urllib.request

# Install cloudflared if needed
if not os.path.exists('/usr/local/bin/cloudflared'):
    subprocess.run('curl -fsSL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared', shell=True, check=True)
print(subprocess.run(['cloudflared', '--version'], capture_output=True, text=True).stdout.strip())

# Restart cleanly if re-run
for _n in ('TUNNEL_PROC', 'PROXY_PROC'):
    _p = globals().get(_n)
    if _p:
        _p.terminate()

# Start proxy (no PROXY_API_KEY - anyone with the URL can use it)
env2 = dict(os.environ)
env2['PATH'] += os.pathsep + os.path.expanduser('~/.bun/bin')
env2.pop('PROXY_API_KEY', None)
PROXY_PROC = subprocess.Popen(['bun', 'run', 'proxy.ts'], cwd='/content/opencode-proxy', env=env2,
                              stdout=open('/tmp/proxy.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print('local health:', urllib.request.urlopen('http://127.0.0.1:8787/health', timeout=2).read().decode())
        break
    except Exception:
        time.sleep(1)

# Start quick tunnel and scrape the public URL from its log
TUNNEL_PROC = subprocess.Popen(['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8787', '--no-autoupdate'],
                               stdout=open('/tmp/tunnel.log', 'w'), stderr=subprocess.STDOUT)
PUBLIC_URL = None
for _ in range(90):
    time.sleep(1)
    try:
        m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/tmp/tunnel.log').read())
    except Exception:
        continue
    if m:
        PUBLIC_URL = m.group(0)
        break
print('PUBLIC_URL =', PUBLIC_URL)
print('Client base URL:', (PUBLIC_URL or '') + '/v1')

## 4 · Option B — zrok (fixed URL, recommended)

Same proxy, but at a **fixed** address every time: `https://<ZROK_NAME>.shares.zrok.io` — the name is reserved to your zrok account permanently (zrok v2: `zrok2 create name` + `share -n public:<name>`).

Needs a free zrok.io account. Two Colab secrets (key icon → Secrets, notebook access on) — never paste them in chat or in a cell:
- `ZROK_TOKEN` — your zrok enable token (required)
- `ZROK_NAME` — your share name (optional; defaults to `oc-llm-proxy`)

Flow: install `zrok2` → enable (secret) → share cell (auto-reclaims the name from old sessions, see §5).

In [ ]:
# Zrok · install zrok2 (v2 CLI) from the latest GitHub release
import json, os, subprocess, tarfile, urllib.request
if not os.path.exists('/usr/local/bin/zrok2'):
    rel = json.load(urllib.request.urlopen('https://api.github.com/repos/openziti/zrok/releases/latest'))
    print('latest release:', rel['tag_name'])
    url = next(a['browser_download_url'] for a in rel['assets'] if 'linux_amd64' in a['name'] and a['name'].endswith('.tar.gz'))
    print('downloading', url)
    urllib.request.urlretrieve(url, '/tmp/zrok.tar.gz')
    tarfile.open('/tmp/zrok.tar.gz').extractall('/tmp/zrok-bin', filter='data')
    for root, _, files in os.walk('/tmp/zrok-bin'):
        for f in files:
            if f.startswith('zrok'):
                src = os.path.join(root, f)
                subprocess.run(['cp', src, '/usr/local/bin/' + f], check=True)
                subprocess.run(['chmod', '+x', '/usr/local/bin/' + f], check=True)
                print('installed', f)
print(subprocess.run(['zrok2', 'version'], capture_output=True, text=True).stdout)

In [ ]:
# Zrok · enable using the ZROK_TOKEN Colab secret (key icon -> Secrets, notebook access ON)
from google.colab import userdata
import subprocess
def zrun(cmd, hide=()):
    p = subprocess.run(cmd, capture_output=True, text=True)
    print('$', ' '.join('***' if c in hide else c for c in cmd))
    print((p.stdout[-1200:] + p.stderr[-600:]).strip())
    return p
try:
    ZROK_TOKEN = userdata.get('ZROK_TOKEN')
except Exception as e:
    ZROK_TOKEN = None
    print('Could not read ZROK_TOKEN secret:', e)
if ZROK_TOKEN:
    zrun(['zrok2', 'enable', '--headless', ZROK_TOKEN], hide=(ZROK_TOKEN,))
else:
    print('No ZROK_TOKEN secret available. Add it under Secrets (key icon) and enable Notebook access, then re-run this cell.')
zrun(['zrok2', 'status'])
zrun(['zrok2', 'list', 'namespaces'])

In [ ]:
# Zrok · share the proxy at your fixed address (enable first)
# Share name: Colab secret ZROK_NAME (Secrets panel), default 'oc-llm-proxy' -> https://<name>.shares.zrok.io
# Self-healing: reclaims the name from any old session's share before binding it to THIS runtime.
import os, re, subprocess, time, urllib.request
from google.colab import userdata
try:
    NAME = (userdata.get('ZROK_NAME') or '').strip() or 'oc-llm-proxy'
except Exception:
    NAME = 'oc-llm-proxy'
print('share name:', NAME, '| client base URL: https://' + NAME + '.shares.zrok.io/v1')
# Make sure the local proxy is up (start it if this runtime was restarted)
try:
    urllib.request.urlopen('http://127.0.0.1:8787/health', timeout=2)
    print('proxy already running')
except Exception:
    env2 = dict(os.environ)
    env2['PATH'] += os.pathsep + os.path.expanduser('~/.bun/bin')
    env2.pop('PROXY_API_KEY', None)
    PROXY_PROC = subprocess.Popen(['bun', 'run', 'proxy.ts'], cwd='/content/opencode-proxy', env=env2,
                                  stdout=open('/tmp/proxy.log', 'w'), stderr=subprocess.STDOUT)
    time.sleep(3)
    print('proxy started')
# Reserve the fixed name once (409 conflict = you already own it, fine)
p = subprocess.run(['zrok2', 'create', 'name', '-n', 'public', NAME], capture_output=True, text=True)
print('create name:', (p.stdout + p.stderr)[-300:].strip())
# Stop our local share process if running
_old = globals().get('ZROK_PROC')
if _old:
    _old.terminate()
    time.sleep(2)
# RECLAIM: delete any share (e.g. from a dead/quota-burned old session) still holding the name
p = subprocess.run(['zrok2', 'list', 'shares'], capture_output=True, text=True)
for line in (p.stdout + p.stderr).splitlines():
    if NAME in line:
        parts = [c.strip() for c in line.split('\u2502') if c.strip()]
        if len(parts) >= 3:
            tok, envz = parts[0], parts[2]
            d = subprocess.run(['zrok2', 'delete', 'share', tok, '--envzid', envz], capture_output=True, text=True)
            print('reclaimed name from old share', tok, (d.stdout + d.stderr)[-200:].strip())
# Start the share bound to THIS runtime: --open = anyone with the URL can use it
ZROK_PROC = subprocess.Popen(['zrok2', 'share', 'public', '--headless', '--open', '-n', 'public:' + NAME, 'localhost:8787'],
                             stdout=open('/tmp/zrok.log', 'w'), stderr=subprocess.STDOUT)
ZROK_URL = None
for _ in range(60):
    time.sleep(1)
    try:
        txt = open('/tmp/zrok.log').read()
    except Exception:
        continue
    m = re.search(r'[a-z0-9.-]*shares?\.zrok\.io', txt)
    if m:
        ZROK_URL = 'https://' + m.group(0)
        break
print('ZROK_URL =', ZROK_URL)
print('Client base URL:', (ZROK_URL or '') + '/v1')
print('log tail:', open('/tmp/zrok.log').read()[-600:])

## 5 · Ops — recovery & shutdown

**Quota burned?** The tunnel points at the proxy process *inside one runtime* — a dead runtime = dead URL, even though your share name (secret `ZROK_NAME`, default `oc-llm-proxy`) stays reserved to your account. Recovery:

1. **New runtime** (Runtime → Restart, or a fresh session) = new egress IP = fresh free quota.
2. Re-run setup: §1 cells → zrok install → enable (your Colab secrets persist).
3. Run the **§4 share cell**: it starts the proxy here, **reclaims the name** from the old session's share if it's still holding it, and binds your fixed URL to this runtime. Client configs never change.

Don't run the share cell in two notebooks at once — they'll fight over the name. **Shutdown:** the last cell stops Cloudflare, zrok, and the proxy.

In [ ]:
# Ops · Stop everything (Cloudflare tunnel, zrok share, proxy) - public URLs stop working
for _n in ('TUNNEL_PROC', 'ZROK_PROC', 'PROXY_PROC'):
    _p = globals().get(_n)
    if _p:
        _p.terminate()
        print(_n, 'stopped')